# Notebook 03 — Advanced Cardiotoxicity: hERG + CiPA Framework
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

Cardiac safety is a critical drug development hurdle. The **CiPA (Comprehensive in vitro Proarrhythmia Assay)** initiative (FDA/HESI/CSRC) moves beyond single-channel hERG blockade to a multi-ion-channel framework. This connects directly to my *Chemistry 2022* paper on hERG prediction.

| Channel | Gene | Cardiac role |
|---------|------|--------------|
| hERG (IKr) | KCNH2 | Repolarization — primary arrhythmia risk |
| Nav1.5 (INa) | SCN5A | Depolarization |
| Cav1.2 (ICaL) | CACNA1C | Plateau phase |

Covers: hERG IC50 regression, CiPA multi-channel risk scoring, GAT-inspired features, Monte Carlo Dropout uncertainty.

In [ ]:
!pip install rdkit scikit-learn torch pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
import torch, torch.nn as nn, torch.nn.functional as F
import warnings; warnings.filterwarnings('ignore')

# CiPA pilot study reference compounds (Crumb et al. 2016)
# hERG_IC50, Nav15_IC50, CaL_IC50 all in uM
cipa = [
    ("OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1",
     "Terfenadine", 0.09, 16.0, 7.4, "High"),
    ("CCOC(=O)c1cc2cc(OC)c(OC)cc2[nH]1","Cisapride",0.012,100,100,"High"),
    ("CN(CCOc1ccc(NS(=O)(=O)c2ccc(NC)cc2)cc1)S(=O)(=O)c1ccc(N)cc1","Dofetilide",0.004,100,100,"High"),
    ("CC(O)CNc1ccc(NS(C)(=O)=O)cc1","Sotalol",127,100,100,"Low"),
    ("COc1ccc(CCN(C)CCC(C#N)(c2ccc(OC)c(OC)c2)C(C)C)cc1OC","Verapamil",0.143,5.0,0.22,"Medium"),
    ("CC(Nc1c(C)cccc1C)COC","Mexiletine",100,0.25,100,"Low"),
    ("OC(c1ccnc2ccccc12)C1CC2CCN1CC2C=C","Quinidine",0.294,5.5,32.0,"High"),
    ("OCC(NC(=O)c1nc2cc(OCC(F)(F)F)ccc2c(OCC(F)(F)F)c1)C","Flecainide",1.09,0.18,40.0,"Medium"),
    ("COc1ccc2c(c1)CC(=O)N(CCN(C)C)c2c1ccc(OC)cc1","Diltiazem",5.74,100,0.069,"Low"),
    ("CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc21","Chlorpromazine",0.185,3.8,3.0,"High"),
    ("Cn1cc2c(cn1)CC(=O)N2CC1CCNCC1","Ondansetron",1.57,100,100,"Low"),
    ("COc1ccc(OCC(O)CN2CC(=O)N(c3ccccc3F)CC2)cc1OC","Ranolazine",11.5,7.5,3.0,"Medium"),
]

def featurize(smi, n_bits=2048):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    fp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,n_bits))
    pc=np.array([
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol), rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol), rdMolDescriptors.CalcNumAromaticRings(mol),
        Descriptors.FractionCSP3(mol), Descriptors.MolMR(mol),
        rdMolDescriptors.CalcNumRings(mol), Descriptors.NumValenceElectrons(mol),
    ])
    return np.concatenate([fp,pc])

X=np.array([featurize(s) for s,*_ in cipa])
names=[r[1] for r in cipa]
y_herg=np.log10([r[2] for r in cipa])
y_nav =np.log10([r[3] for r in cipa])
y_cal =np.log10([r[4] for r in cipa])
y_risk=[r[5] for r in cipa]
scaler=StandardScaler(); X_s=scaler.fit_transform(X)
print(f"CiPA dataset: {len(cipa)} reference compounds")
print(f"hERG IC50 range: {10**y_herg.min():.3f} to {10**y_herg.max():.1f} uM")

## hERG IC50 Regression (quantitative safety pharmacology)

In [ ]:
kf=KFold(4,shuffle=True,random_state=42)
for nm,reg in [("Random Forest",RandomForestRegressor(300,random_state=42)),
               ("SVR (RBF)",SVR(kernel='rbf',C=10,gamma='scale'))]:
    r2=cross_val_score(reg,X_s,y_herg,cv=kf,scoring='r2')
    mse=cross_val_score(reg,X_s,y_herg,cv=kf,scoring='neg_mean_squared_error')
    print(f"{nm:20s}  R2={r2.mean():.3f}  RMSE={(-mse.mean())**0.5:.3f} log10-uM")

rf=RandomForestRegressor(300,random_state=42); rf.fit(X_s,y_herg)
yp=rf.predict(X_s)
rc={"High":"#e74c3c","Medium":"#f39c12","Low":"#27ae60"}
fig,ax=plt.subplots(figsize=(7,6))
for name,yt,ypr,risk in zip(names,y_herg,yp,y_risk):
    ax.scatter(yt,ypr,c=rc[risk],s=100,zorder=5)
    ax.annotate(name,(yt,ypr),fontsize=7,xytext=(3,3),textcoords='offset points')
xl=np.linspace(y_herg.min()-0.5,y_herg.max()+0.5,100)
ax.plot(xl,xl,'k--',lw=1,label='Perfect')
for r,c in rc.items(): ax.scatter([],[],c=c,label=f"CiPA {r}",s=60)
ax.set_xlabel("log10(hERG IC50 uM) Observed"); ax.set_ylabel("Predicted")
ax.set_title("hERG IC50 Regression"); ax.legend()
plt.tight_layout(); plt.savefig("herg_regression.png",dpi=150); plt.show()

## CiPA Multi-channel Risk Scoring

In [ ]:
def cipa_risk(herg, nav, cal):
    herg_b = 1/(1+herg/0.3)
    nav_b  = 1/(1+nav/1.0)
    cal_b  = 1/(1+cal/1.0)
    net    = herg_b - 0.5*nav_b - 0.5*cal_b
    if net>0.5:   return "High",net
    elif net>0.2: return "Medium",net
    else:         return "Low",net

print(f"{'Compound':15s} {'hERG':>10} {'Nav':>10} {'CaL':>10} {'Pred':>8} {'True':>8} {'Match'}")
print("-"*75)
correct=0
for r in cipa:
    s,nm,h,nv,cl,true=r
    pred,score=cipa_risk(h,nv,cl)
    match="OK" if pred==true else "X"
    if pred==true: correct+=1
    print(f"{nm:15s} {h:10.3f} {nv:10.1f} {cl:10.1f} {pred:>8} {true:>8} {match}")
print(f"Accuracy: {correct}/{len(cipa)} = {correct/len(cipa)*100:.0f}%")

## Monte Carlo Dropout — uncertainty quantification

In [ ]:
class MCDropNet(nn.Module):
    def __init__(self,d,drop=0.3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(d,256),nn.ReLU(),nn.Dropout(drop),
            nn.Linear(256,128),nn.ReLU(),nn.Dropout(drop),
            nn.Linear(128,1))
    def forward(self,x): return self.net(x)

Xt=torch.FloatTensor(X_s); yt=torch.FloatTensor(y_herg).unsqueeze(1)
mc=MCDropNet(X_s.shape[1]); opt=torch.optim.Adam(mc.parameters(),lr=1e-3)
for ep in range(100):
    mc.train(); opt.zero_grad()
    F.mse_loss(mc(Xt),yt).backward(); opt.step()

mc.train()  # keep dropout active
preds=np.array([mc(Xt).detach().numpy().flatten() for _ in range(50)])
mu=preds.mean(0); sigma=preds.std(0)

fig,ax=plt.subplots(figsize=(10,4))
rc2={"High":"#e74c3c","Medium":"#f39c12","Low":"#27ae60"}
cols=[rc2[r] for r in y_risk]
ax.bar(range(len(names)),10**mu,yerr=10**sigma,capsize=4,color=cols,alpha=0.8)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names,rotation=45,ha='right',fontsize=8)
ax.set_ylabel("hERG IC50 (uM) +/- uncertainty"); ax.set_yscale('log')
ax.set_title("MC Dropout hERG Predictions with Uncertainty")
ax.axhline(1,color='k',linestyle='--',lw=0.8,label='1 uM threshold'); ax.legend()
plt.tight_layout(); plt.savefig("mc_dropout_herg.png",dpi=150); plt.show()
print("High uncertainty compounds (flag for experiment):")
for n,m,s in sorted(zip(names,mu,sigma),key=lambda x:-x[2])[:3]:
    print(f"  {n}: log10(IC50)={m:.2f} +/- {s:.2f}")

## Key takeaways
- hERG blockade alone is insufficient — CiPA integrates Nav1.5 and Cav1.2 balance
- Quantitative IC50 regression is more actionable than binary classification for lead opt
- MC Dropout provides calibrated uncertainty — flag high-sigma predictions for wet lab
- Verapamil is a hERG blocker but clinically safe because Nav1.5/Cav1.2 balance it
- Regulatory: ICH E14 (clinical QT), ICH S7B (non-clinical cardiac), FDA CiPA framework
- Industry tools: hERGBoost, AttenhERG, CardioTox (web accessible)